In [4]:
# file: sensoroutput_sim.py
# Benchmark labeler (causal 4s window):
# Priority per timestep t using causal window [t-3..t]:
#   - Frozen Sensor overwrites ALL labels in the window (including current Error Value).
#   - If current row has its own error and Frozen is not active, keep it.
#   - If current row is non-error, window overwrites with hierarchy:
#       Error Value  >  Uncalibrated
# Frozen uses pairwise exact-equality deltas for (t-1,t) ONLY,
# and we remember the last THREE pair flags so the 4s window can light up from:
#   (t-4,t-3), (t-3,t-2), (t-2,t-1)  plus the current (t-1,t).
# ΔRPM behaviour unchanged.
#
# CHANGE (v2.1): ForcedWarmup no longer hardcodes "Engine Off (cold)".
# It now uses the same temperature rule as the normal Off branch
# (cold < 50, warm 50-120, generic otherwise). Runtime has the
# IDENTICAL change so benchmark and runtime stay matched.

import os
import numpy as np
import pandas as pd
from collections import deque

# ========= Paths =========
X_in_path   = "../../../data/simulation/engine_total_X.npy"
out_dir     = "../../../product/io/Test/"
os.makedirs(out_dir, exist_ok=True)

X_out_path  = os.path.join(out_dir, "engine_total_X.npy")
Y_out_path  = os.path.join(out_dir, "engine_total_benchmark_y.npy")
CSV_out_path= os.path.join(out_dir, "engine_total_benchmark.csv")

RPM_IDX = 2

# Toggles
CONTIGUOUS_BEHAVIOUR                   = True
FROZEN_USE_PREV_TAIL_ROW_AT_T0         = True
FROZEN_CARRY_LAST3_PAIR_FLAGS_ACROSS   = True
ROW_ERR_CROSS_SEQUENCE                 = True

# ---------- helpers (numeric-safe) ----------
def _to_float_or_nan(x):
    try:
        f = float(x)
        return f if np.isfinite(f) else np.nan
    except Exception:
        return np.nan


def _row_to_float64(row4):
    return np.array([_to_float_or_nan(row4[i]) for i in range(4)], dtype=np.float64)


def in_range(x: float, a: float, b: float) -> bool:
    return (x >= a) and (x <= b)


def _off_label_from_temp(temp: float) -> str:
    """Same temperature rule as the normal Off branch (single source of truth
    for warmup labels). cold < 50, warm 50-120, generic otherwise."""
    if np.isfinite(temp) and in_range(temp, -20, 49.999999):
        return "Engine Off (cold)"
    if np.isfinite(temp) and in_range(temp, 50, 120):
        return "Engine Off (warm)"
    return "Engine Off"


def _pair_is_frozen(prev_vals: np.ndarray | None, curr_vals: np.ndarray | None) -> bool:
    if prev_vals is None or curr_vals is None:
        return False

    epsilon = 1e-8
    pv, cv = prev_vals, curr_vals
    finite = np.isfinite(pv) & np.isfinite(cv)
    if not finite.any():
        return False

    for j in (0, 1, 3):
        if finite[j] and abs(cv[j] - pv[j]) <= epsilon:
            return True

    if finite[RPM_IDX] and (pv[RPM_IDX] != 0.0) and (cv[RPM_IDX] != 0.0):
        if abs(cv[RPM_IDX] - pv[RPM_IDX]) <= epsilon:
            return True

    return False

def _csv_val(v):
    if isinstance(v, (np.floating, float, int, np.integer)):
        return float(v)
    try:
        return float(v)
    except Exception:
        return str(v)


# ---------- row-level error ----------
def detect_row_reason(row4: np.ndarray) -> str:
    # GROUND TRUTH detector: covers BOTH the extreme zones (which the
    # runtime's rule check also catches) and the MODERATE zones (which
    # pass the runtime rules and must be caught by M0). The runtime's
    # own _row_reason keeps ONLY the extreme ranges -- that asymmetry
    # is intentional: it is what makes M0's detection measurable.
    vals = _row_to_float64(row4)
    if not np.isfinite(vals).all():
        return "Error Value"

    t, p, r, v = vals

    # Temperature: extreme + moderate
    if (-90.0 <= t <= -30.0) or (165.0 <= t <= 300.0):
        return "Uncalibrated"
    if (-29.9 <= t <= -21.0) or (146.0 <= t <= 164.9):
        return "Uncalibrated"

    # Pressure: extreme + moderate
    if (-6.0 <= p <= 0.01) or (1.5 <= p <= 6.0):
        return "Uncalibrated"
    if (0.05 <= p <= 0.45) or (1.30 <= p <= 1.45):
        return "Uncalibrated"

    # RPM: extreme + moderate
    if (-2000.0 <= r <= -0.1) or (12000.0 <= r <= 100000.0):
        return "Uncalibrated"
    if (9500.0 <= r <= 11900.0):
        return "Uncalibrated"

    # Vibration: extreme + moderate
    if (-20.0 <= v <= -0.1) or (2.0 <= v <= 20.0):
        return "Uncalibrated"
    if (0.80 <= v <= 1.90):
        return "Uncalibrated"

    return ""


# ---------- main labeler ----------
global_warmup_counter = 0
global_history_rpm_prev = None

def label_sequence(
    X_seq: np.ndarray,
    prev_tail_row: np.ndarray | None,
    prev_tail_pair_frozen_flags: np.ndarray | None,
    prev_tail_row_reasons: np.ndarray | None
):
    X_seq = np.asarray(X_seq, dtype=object)
    T = int(X_seq.shape[0])

    labels, rows = [], []
    global global_warmup_counter
    global global_history_rpm_prev
    # --- Runtime-matching warmup state ---

    last_pairs_frozen = deque(maxlen=3)
    if FROZEN_CARRY_LAST3_PAIR_FLAGS_ACROSS and prev_tail_pair_frozen_flags is not None:
        for b in list(prev_tail_pair_frozen_flags)[-3:]:
            last_pairs_frozen.append(bool(b))

    last_row_reasons = deque(maxlen=3)
    if ROW_ERR_CROSS_SEQUENCE and prev_tail_row_reasons is not None:
        for s in list(prev_tail_row_reasons)[-3:]:
            last_row_reasons.append(str(s))

    prev_vals_numeric = _row_to_float64(prev_tail_row[:4]) if (prev_tail_row is not None) else None

    for t in range(T):
        raw_temp, raw_pres, raw_rpm, raw_vib = X_seq[t, 0], X_seq[t, 1], X_seq[t, 2], X_seq[t, 3]
        vals_t = _row_to_float64(X_seq[t, :4])
        temp, pres, rpm, vib = vals_t
        row_reason = detect_row_reason(X_seq[t, :4])
        # --- Engine reset detection (match runtime) ---
        if global_history_rpm_prev is not None:
         if np.isfinite(global_history_rpm_prev) and np.isfinite(rpm):
           if global_history_rpm_prev > 0 and rpm == 0:
            last_pairs_frozen.clear()
            last_row_reasons.clear()
            global_warmup_counter = 0

        # --- Forced warmup (first 4 samples after reset or sequence start) ---
        if global_warmup_counter < 4:
            global_warmup_counter += 1

            # CHANGED: temperature-based label instead of hardcoded cold
            label = _off_label_from_temp(temp)
            rows.append([
                t,
                _csv_val(raw_temp),
                _csv_val(raw_pres),
                _csv_val(raw_rpm),
                _csv_val(raw_vib),
                label,
                "",
                False,
                False,
                "ForcedWarmup",
                ""
            ])
            labels.append(label)

            global_history_rpm_prev = rpm
            last_pairs_frozen.append(False)
            last_row_reasons.append(row_reason)
            continue

        if t == 0 and FROZEN_USE_PREV_TAIL_ROW_AT_T0:
            prev_for_pair = prev_vals_numeric
        else:
            prev_for_pair = _row_to_float64(X_seq[t-1, :4]) if t > 0 else None
        pair_frozen = _pair_is_frozen(prev_for_pair, vals_t)

        window_row_reasons = list(last_row_reasons) + [row_reason]
        has_errval_window  = any(r == "Error Value"  for r in window_row_reasons)
        has_uncal_window   = any(r == "Uncalibrated" for r in window_row_reasons)

        window_frozen_flag = pair_frozen or any(last_pairs_frozen)

        # ---- Runtime-matching priority ----
        if has_errval_window:
         label = "Unknown [Error Value]"

        elif window_frozen_flag:
         label = "Unknown [Frozen Sensor]"

        elif has_uncal_window:
         label = "Unknown [Uncalibrated]"

        else:
          label = ""

        if label:
            if t == 0:
                dRPM_dbg = ""
            else:
                prev_r = _row_to_float64(X_seq[t-1, :4])[RPM_IDX]
                dRPM_dbg = float(rpm - prev_r) if (np.isfinite(rpm) and np.isfinite(prev_r)) else ""
            rows.append([t, _csv_val(raw_temp), _csv_val(raw_pres), _csv_val(raw_rpm), _csv_val(raw_vib),
                         label, dRPM_dbg, True, (row_reason != ""), "win", ""])
            labels.append(label)
        else:
            if np.isfinite(rpm) and rpm == 0.0:
                lab = _off_label_from_temp(temp)
                if t == 0:
                    dRPM_dbg = ""
                else:
                    prev_r = _row_to_float64(X_seq[t-1, :4])[RPM_IDX]
                    dRPM_dbg = float(rpm - prev_r) if np.isfinite(prev_r) else ""
                rows.append([t, _csv_val(raw_temp), _csv_val(raw_pres), _csv_val(raw_rpm), _csv_val(raw_vib),
                             lab, dRPM_dbg, False, False, "off", ""])
                labels.append(lab)
            else:
                a = max(0, t - 3)
                rpm_slice = [_row_to_float64(X_seq[i, :4])[RPM_IDX] for i in range(a, t+1)]
                has_zero_numeric = any(np.isfinite(rv) and rv == 0.0 for rv in rpm_slice)
                if has_zero_numeric:
                    lab = "Engine Start"
                    if t == 0:
                        dRPM_dbg = ""
                    else:
                        prev_r = _row_to_float64(X_seq[t-1, :4])[RPM_IDX]
                        dRPM_dbg = float(rpm - prev_r) if (np.isfinite(rpm) and np.isfinite(prev_r)) else ""
                    rows.append([t, _csv_val(raw_temp), _csv_val(raw_pres), _csv_val(raw_rpm), _csv_val(raw_vib),
                                 lab, dRPM_dbg, False, False, "start", ""])
                    labels.append(lab)
                else:
                    dRPM_dbg = ""
                    prev_rpm_val = None
                    if t == 0:
                        if CONTIGUOUS_BEHAVIOUR and (prev_tail_row is not None):
                            prev_rpm_val = _row_to_float64(prev_tail_row[:4])[RPM_IDX]
                            if np.isfinite(rpm) and np.isfinite(prev_rpm_val):
                                dRPM_dbg = float(rpm - prev_rpm_val)
                    else:
                        prev_rpm_val = _row_to_float64(X_seq[t-1, :4])[RPM_IDX]
                        if np.isfinite(rpm) and np.isfinite(prev_rpm_val):
                            dRPM_dbg = float(rpm - prev_rpm_val)

                    if not np.isfinite(prev_rpm_val) or not np.isfinite(rpm):
                        behaviour = "idle"; src = "none" if t == 0 else "intra"
                    else:
                        if dRPM_dbg >= 40.0:
                            behaviour = "accelerating"; src = "contig" if t==0 and CONTIGUOUS_BEHAVIOUR else "intra"
                        elif dRPM_dbg <= -40.0:
                            behaviour = "decelerating"; src = "contig" if t==0 and CONTIGUOUS_BEHAVIOUR else "intra"
                        else:
                            behaviour = "idle"; src = "contig" if t==0 and CONTIGUOUS_BEHAVIOUR else "intra"

                    if np.isfinite(temp) and in_range(temp, 80, 105):        level = "NormalLoad"
                    elif np.isfinite(temp) and in_range(temp, 105, 115):     level = "HighLoad"
                    elif np.isfinite(temp) and in_range(temp, 115, 145):     level = "CriticalLoad"
                    else:                                                    level = "NormalLoad"

                    lab = f"{level} ({behaviour})"
                    rows.append([t, _csv_val(raw_temp), _csv_val(raw_pres), _csv_val(raw_rpm), _csv_val(raw_vib),
                                 lab, dRPM_dbg, False, False, src, ""])
                    labels.append(lab)

        last_pairs_frozen.append(bool(pair_frozen))
        last_row_reasons.append(row_reason)
        prev_vals_numeric = vals_t
        global_history_rpm_prev = rpm

    last_row               = np.array([X_seq[-1, i] for i in range(4)], dtype=object)
    last_three_pair_flags  = np.array(list(last_pairs_frozen)[-3:], dtype=bool)
    last_three_reasons     = np.array(list(last_row_reasons)[-3:], dtype=object)
    return labels, rows, last_row, last_three_pair_flags, last_three_reasons


# ========= Load X =========
X = np.load(X_in_path, allow_pickle=True)
if X.dtype != object:
    X = np.asarray([np.asarray(s, dtype=object) for s in X], dtype=object)


# ========= Generate Y + CSV =========
Y_list, rows_all = [], []
prev_tail_row = None
prev_tail_pair_frozen_flags = None
prev_tail_row_reasons = None

for seq_idx, X_seq in enumerate(X):
    labels, rows, prev_tail_row, prev_tail_pair_frozen_flags, prev_tail_row_reasons = label_sequence(
        X_seq,
        prev_tail_row if (CONTIGUOUS_BEHAVIOUR or FROZEN_USE_PREV_TAIL_ROW_AT_T0) else None,
        prev_tail_pair_frozen_flags if FROZEN_CARRY_LAST3_PAIR_FLAGS_ACROSS else None,
        prev_tail_row_reasons if ROW_ERR_CROSS_SEQUENCE else None
    )
    for r in rows:
        t, temp, pres, rpm, vib, lab, dRPM_dbg, win_issue, base_issue, source, reason = r
        rows_all.append([seq_idx, t, temp, pres, rpm, vib, lab, dRPM_dbg, win_issue, base_issue, source, reason])
    Y_list.append(np.asarray(labels, dtype=object))


# ========= Save outputs =========
np.save(X_out_path, np.asarray([np.asarray(s, dtype=object) for s in X], dtype=object), allow_pickle=True)
np.save(Y_out_path, np.asarray(Y_list, dtype=object), allow_pickle=True)

pd.DataFrame(
    rows_all,
    columns=["sequence","timestep","Temperature","Pressure","RPM","Vibration","Label","dRPM",
             "WindowUnknown","BaseUnknown","source","Reason"]
).to_csv(CSV_out_path, index=False)

print("DONE")

DONE
